# Explore here

In [1]:
# Your code here

import pandas as pd

# Cargar el dataset desde la URL oficial
url = "https://breathecode.herokuapp.com/asset/internal-link?id=435&path=url_spam.csv"
df = pd.read_csv(url)

# Mostrar la forma del dataset y las primeras 5 filas
print(f"Dimensiones del dataset: {df.shape}\n")
df.head()

Dimensiones del dataset: (2999, 2)



,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,True
1,https://www.hvper.com/,True
2,https://briefingday.com/m/v4n3i4f3,True
3,https://briefingday.com/n/20200618/m#commentform,False
4,https://briefingday.com/fan,True


In [3]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

def clean_url(url):
    # Reemplazar signos de puntuación por espacios para separar la URL
    url = re.sub(r'[^\w\s]', ' ', str(url))
    # Convertir a minúsculas y quitar espacios extra
    return url.lower()

# Preprocesar la columna de URLs
df['clean_url'] = df['url'].apply(clean_url)

# Dividir dataset en Train (80%) y Test (20%)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_url'], 
    df['is_spam'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['is_spam']
)

# Convertir las URLs procesadas a vectores numéricos TF-IDF con palabras vacías en inglés incorporadas
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")

Dimensiones de X_train: (2399, 5000)
Dimensiones de X_test: (600, 5000)


In [4]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Crear y entrenar el modelo SVM por defecto
svm_base = SVC(random_state=42)
svm_base.fit(X_train, y_train)

# Realizar predicciones sobre el conjunto de prueba
y_pred_base = svm_base.predict(X_test)

# Evaluar el modelo
print("=== Resultados SVM Base ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.4f}\n")
print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred_base))

=== Resultados SVM Base ===
Accuracy: 0.9533

Reporte de Clasificación:
              precision    recall  f1-score   support

       False       0.96      0.98      0.97       461
        True       0.93      0.86      0.90       139

    accuracy                           0.95       600
   macro avg       0.94      0.92      0.93       600
weighted avg       0.95      0.95      0.95       600



In [5]:
from sklearn.model_selection import GridSearchCV

# Definir la cuadrícula de parámetros a probar
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

# Configurar la búsqueda en cuadrícula con validación cruzada (3-folds)
grid_search = GridSearchCV(
    estimator=SVC(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Obtener el mejor modelo
best_svm = grid_search.best_estimator_

print("Mejores hiperparámetros encontrados:")
print(grid_search.best_params_)

# Evaluar el modelo optimizado en Test
y_pred_opt = best_svm.predict(X_test)

print("\n=== Resultados SVM Optimizado ===")
print(f"Accuracy Óptimo: {accuracy_score(y_test, y_pred_opt):.4f}\n")
print("Reporte de Clasificación Óptimo:")
print(classification_report(y_test, y_pred_opt))

Mejores hiperparámetros encontrados:
{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}

=== Resultados SVM Optimizado ===
Accuracy Óptimo: 0.9600

Reporte de Clasificación Óptimo:
              precision    recall  f1-score   support

       False       0.97      0.98      0.97       461
        True       0.93      0.90      0.91       139

    accuracy                           0.96       600
   macro avg       0.95      0.94      0.94       600
weighted avg       0.96      0.96      0.96       600



In [ ]:
import joblib
import os

# Carpeta models
os.makedirs("../models", exist_ok=True)

# Guardar el modelo y el vectorizador
joblib.dump(best_svm, "../models/svm_spam_model.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")

print("¡Modelo SVM y Vectorizador guardados exitosamente en la carpeta 'models/'!")

¡Modelo SVM y Vectorizador guardados exitosamente en la carpeta 'models/'!
